# PRMP vs Attention-Based Aggregation (GATConv) Comparison

**Experiment:** Compare 4 heterogeneous GNN variants on Amazon Video Games review-rating regression:
- **(A) HeteroSAGEConv** — mean aggregation baseline
- **(B) HeteroGATConv** — attention-weighted aggregation
- **(C) PRMP+SAGE** — predict-subtract residual with mean aggregation
- **(D) PRMP+GAT** — predict-subtract residual with attention aggregation

All GNN layers are implemented in **pure PyTorch** (no torch-geometric) to avoid torch-scatter/torch-sparse compilation issues.

This notebook:
1. Defines all 4 GNN model architectures
2. Demonstrates forward passes on a small synthetic heterogeneous graph
3. Trains on synthetic data to show the training pipeline
4. Visualizes pre-computed experiment results (50K reviews, 3 seeds × 200 epochs)

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')
    _pip('torch', '--extra-index-url', 'https://download.pytorch.org/whl/cpu')

In [ ]:
import copy
import gc
import itertools
import json
import math
import os
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/experiment_iter5_prmp_vs_attenti/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data['datasets'][0]['examples'])} examples")
print(f"Metadata keys: {list(data['metadata'].keys())}")
print(f"Variants tested: {list(data['metadata']['results_summary'].keys())}")

## Configuration

Tunable parameters for the synthetic demo graph and training. These are set to small values for a quick demo; the original experiment used much larger values (commented).

In [ ]:
# --- Tunable demo parameters ---
HIDDEN_DIM = 32         # Original: 128
DROPOUT = 0.3           # Original: 0.3
LR = 0.001              # Original: 0.001
WEIGHT_DECAY = 1e-5     # Original: 1e-5
MAX_EPOCHS = 5          # Original: 200
PATIENCE = 3            # Original: 20
GAT_HEADS = 2           # Original: 4
DEMO_SEED = 42

# Synthetic graph sizes (demo only — original used 50K reviews, 3.4K products, 10.9K customers)
N_DEMO_REVIEWS = 50
N_DEMO_PRODUCTS = 5
N_DEMO_CUSTOMERS = 10

# Feature dimensions (must match model architecture)
REVIEW_FEAT_DIM = 21    # Fixed: 3 time + 16 hash + 2 helpful
PRODUCT_FEAT_DIM = 5    # Fixed: mean_r, std_r, cnt, mean_hu, mean_ht
CUSTOMER_FEAT_DIM = 5   # Fixed: same as product

print(f"Config: hidden={HIDDEN_DIM}, epochs={MAX_EPOCHS}, "
      f"GAT heads={GAT_HEADS}, demo graph: {N_DEMO_REVIEWS} reviews, "
      f"{N_DEMO_PRODUCTS} products, {N_DEMO_CUSTOMERS} customers")

## Pure PyTorch GNN Layers

All message-passing layers are implemented from scratch using only PyTorch's `scatter_add_` — no torch-geometric dependency.

- **scatter_mean / scatter_add**: Custom aggregation primitives
- **BipartiteSAGEConv**: Mean-aggregation message passing for bipartite edges
- **BipartiteGATConv**: Multi-head attention message passing for bipartite edges
- **PRMPSAGEConv**: Predict-subtract residual + mean aggregation (the PRMP mechanism)
- **PRMPGATConv**: Predict-subtract residual + attention aggregation

In [ ]:
def scatter_mean(src, index, dim_size):
    """Scatter mean aggregation: aggregate src by index, returning mean per target."""
    out = torch.zeros(dim_size, src.size(1), device=src.device, dtype=src.dtype)
    count = torch.zeros(dim_size, 1, device=src.device, dtype=src.dtype)
    out.scatter_add_(0, index.unsqueeze(1).expand_as(src), src)
    ones = torch.ones(src.size(0), 1, device=src.device, dtype=src.dtype)
    count.scatter_add_(0, index.unsqueeze(1), ones)
    count = count.clamp(min=1)
    return out / count


def scatter_add_fn(src, index, dim_size):
    """Scatter add aggregation."""
    out = torch.zeros(dim_size, src.size(1), device=src.device, dtype=src.dtype)
    out.scatter_add_(0, index.unsqueeze(1).expand_as(src), src)
    return out


class BipartiteSAGEConv(nn.Module):
    """SAGEConv for bipartite edges: aggregates src features to dst nodes."""
    def __init__(self, src_dim, dst_dim, out_dim):
        super().__init__()
        self.lin_neigh = nn.Linear(src_dim, out_dim)
        self.lin_self = nn.Linear(dst_dim, out_dim)

    def forward(self, x_src, x_dst, edge_src, edge_dst, num_dst):
        neigh_feats = x_src[edge_src]
        agg = scatter_mean(neigh_feats, edge_dst, num_dst)
        return self.lin_neigh(agg) + self.lin_self(x_dst)


class BipartiteGATConv(nn.Module):
    """GATConv for bipartite edges with multi-head attention."""
    def __init__(self, src_dim, dst_dim, out_dim, heads=GAT_HEADS, concat=True):
        super().__init__()
        self.heads = heads
        self.out_per_head = out_dim // heads if concat else out_dim
        self.concat = concat
        self.lin_src = nn.Linear(src_dim, heads * self.out_per_head, bias=False)
        self.lin_dst = nn.Linear(dst_dim, heads * self.out_per_head, bias=False)
        self.att_src = nn.Parameter(torch.randn(1, heads, self.out_per_head) * 0.01)
        self.att_dst = nn.Parameter(torch.randn(1, heads, self.out_per_head) * 0.01)
        self.lin_self = nn.Linear(dst_dim, heads * self.out_per_head if concat else out_dim)

    def forward(self, x_src, x_dst, edge_src, edge_dst, num_dst):
        H, D = self.heads, self.out_per_head
        src_feat = self.lin_src(x_src).view(-1, H, D)
        dst_feat = self.lin_dst(x_dst).view(-1, H, D)
        src_edge = src_feat[edge_src]
        dst_edge = dst_feat[edge_dst]
        alpha = F.leaky_relu((src_edge * self.att_src).sum(-1) + (dst_edge * self.att_dst).sum(-1), 0.2)
        alpha = self._edge_softmax(alpha, edge_dst, num_dst)
        msg = (src_edge * alpha.unsqueeze(-1)).view(-1, H * D)
        agg = scatter_add_fn(msg, edge_dst, num_dst)
        return agg + self.lin_self(x_dst)

    def _edge_softmax(self, alpha, edge_dst, num_dst):
        alpha_max = torch.full((num_dst, alpha.size(1)), float('-inf'), device=alpha.device)
        alpha_max.scatter_reduce_(0, edge_dst.unsqueeze(1).expand_as(alpha), alpha, reduce="amax", include_self=True)
        alpha = (alpha - alpha_max[edge_dst]).exp()
        alpha_sum = torch.zeros(num_dst, alpha.size(1), device=alpha.device)
        alpha_sum.scatter_add_(0, edge_dst.unsqueeze(1).expand_as(alpha), alpha)
        return alpha / (alpha_sum[edge_dst] + 1e-16)


class PRMPSAGEConv(nn.Module):
    """PRMP: parent predicts child, subtract prediction, aggregate residuals with mean."""
    def __init__(self, src_dim, dst_dim, out_dim):
        super().__init__()
        self.pred_mlp = nn.Sequential(
            nn.Linear(dst_dim, src_dim), nn.ReLU(), nn.Linear(src_dim, src_dim))
        nn.init.zeros_(self.pred_mlp[2].weight)
        nn.init.zeros_(self.pred_mlp[2].bias)
        self.lin_neigh = nn.Linear(src_dim, out_dim)
        self.lin_self = nn.Linear(dst_dim, out_dim)

    def forward(self, x_src, x_dst, edge_src, edge_dst, num_dst):
        pred_src = self.pred_mlp(x_dst)
        residual = x_src[edge_src] - pred_src[edge_dst]  # THE CORE PRMP OPERATION
        agg = scatter_mean(residual, edge_dst, num_dst)
        return self.lin_neigh(agg) + self.lin_self(x_dst)


class PRMPGATConv(nn.Module):
    """PRMP with attention-weighted aggregation of residuals."""
    def __init__(self, src_dim, dst_dim, out_dim, heads=GAT_HEADS, concat=True):
        super().__init__()
        self.heads = heads
        self.out_per_head = out_dim // heads if concat else out_dim
        self.concat = concat
        self.pred_mlp = nn.Sequential(
            nn.Linear(dst_dim, src_dim), nn.ReLU(), nn.Linear(src_dim, src_dim))
        nn.init.zeros_(self.pred_mlp[2].weight)
        nn.init.zeros_(self.pred_mlp[2].bias)
        self.lin_res = nn.Linear(src_dim, heads * self.out_per_head, bias=False)
        self.lin_src = nn.Linear(src_dim, heads * self.out_per_head, bias=False)
        self.lin_dst = nn.Linear(dst_dim, heads * self.out_per_head, bias=False)
        self.att_src = nn.Parameter(torch.randn(1, heads, self.out_per_head) * 0.01)
        self.att_dst = nn.Parameter(torch.randn(1, heads, self.out_per_head) * 0.01)
        self.lin_self = nn.Linear(dst_dim, heads * self.out_per_head if concat else out_dim)

    def forward(self, x_src, x_dst, edge_src, edge_dst, num_dst):
        H, D = self.heads, self.out_per_head
        pred_src = self.pred_mlp(x_dst)
        residual = x_src[edge_src] - pred_src[edge_dst]
        src_feat = self.lin_src(x_src).view(-1, H, D)
        dst_feat = self.lin_dst(x_dst).view(-1, H, D)
        src_edge, dst_edge = src_feat[edge_src], dst_feat[edge_dst]
        alpha = F.leaky_relu((src_edge * self.att_src).sum(-1) + (dst_edge * self.att_dst).sum(-1), 0.2)
        alpha = self._edge_softmax(alpha, edge_dst, num_dst)
        res_transformed = self.lin_res(residual).view(-1, H, D)
        msg = (res_transformed * alpha.unsqueeze(-1)).view(-1, H * D)
        agg = scatter_add_fn(msg, edge_dst, num_dst)
        return agg + self.lin_self(x_dst)

    def _edge_softmax(self, alpha, edge_dst, num_dst):
        alpha_max = torch.full((num_dst, alpha.size(1)), float('-inf'), device=alpha.device)
        alpha_max.scatter_reduce_(0, edge_dst.unsqueeze(1).expand_as(alpha), alpha, reduce="amax", include_self=True)
        alpha = (alpha - alpha_max[edge_dst]).exp()
        alpha_sum = torch.zeros(num_dst, alpha.size(1), device=alpha.device)
        alpha_sum.scatter_add_(0, edge_dst.unsqueeze(1).expand_as(alpha), alpha)
        return alpha / (alpha_sum[edge_dst] + 1e-16)

print("GNN layers defined: BipartiteSAGEConv, BipartiteGATConv, PRMPSAGEConv, PRMPGATConv")

## Heterogeneous GNN Models

Four model variants built from the layers above, each using a 2-layer architecture with heterogeneous message passing across `review ↔ product` and `review ↔ customer` edges:

| Variant | Child→Parent edges | Parent→Child edges |
|---------|-------------------|-------------------|
| HeteroSAGE | SAGEConv (mean) | SAGEConv (mean) |
| HeteroGAT | GATConv (attention) | GATConv (attention) |
| PRMP+SAGE | PRMPSAGEConv (residual+mean) | SAGEConv (mean) |
| PRMP+GAT | PRMPGATConv (residual+attention) | GATConv (attention) |

In [ ]:
# Edge type definitions used by all models
EDGE_TYPES = [
    ("review__rev_of__product", "review", "product",
     "edge_review_to_product_src", "edge_review_to_product_dst", "n_products"),
    ("product__has_review__review", "product", "review",
     "edge_product_to_review_src", "edge_product_to_review_dst", "n_reviews"),
    ("review__written_by__customer", "review", "customer",
     "edge_review_to_customer_src", "edge_review_to_customer_dst", "n_customers"),
    ("customer__wrote__review", "customer", "review",
     "edge_customer_to_review_src", "edge_customer_to_review_dst", "n_reviews"),
]
PRMP_EDGE_KEYS = {"review__rev_of__product", "review__written_by__customer"}


class HeteroGNNLayer(nn.Module):
    """One layer of heterogeneous message passing across all edge types."""
    def __init__(self, conv_dict):
        super().__init__()
        self.convs = nn.ModuleDict(conv_dict)

    def forward(self, x_dict, graph):
        out_dict = {}
        for key, src_type, dst_type, esrc_key, edst_key, ndst_key in EDGE_TYPES:
            if key not in self.convs:
                continue
            result = self.convs[key](x_dict[src_type], x_dict[dst_type],
                                     graph[esrc_key], graph[edst_key], graph[ndst_key])
            if dst_type not in out_dict:
                out_dict[dst_type] = result
            else:
                out_dict[dst_type] = out_dict[dst_type] + result
        return out_dict


def _make_hetero_model(conv_factory, hidden=HIDDEN_DIM):
    """Create a 2-layer heterogeneous GNN with given conv factory."""
    class _Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.input_lins = nn.ModuleDict({
                "product": nn.Linear(PRODUCT_FEAT_DIM, hidden),
                "customer": nn.Linear(CUSTOMER_FEAT_DIM, hidden),
                "review": nn.Linear(REVIEW_FEAT_DIM, hidden),
            })
            self.conv1 = HeteroGNNLayer({k: conv_factory(hidden, hidden, hidden) for k, *_ in EDGE_TYPES})
            self.conv2 = HeteroGNNLayer({k: conv_factory(hidden, hidden, hidden) for k, *_ in EDGE_TYPES})
            self.head = nn.Linear(hidden, 1)
            self.dropout = nn.Dropout(DROPOUT)

        def forward(self, graph):
            x_dict = {
                "product": F.relu(self.input_lins["product"](graph["x_product"])),
                "customer": F.relu(self.input_lins["customer"](graph["x_customer"])),
                "review": F.relu(self.input_lins["review"](graph["x_review"])),
            }
            out = self.conv1(x_dict, graph)
            x_dict = {k: F.relu(self.dropout(v)) for k, v in out.items()}
            out = self.conv2(x_dict, graph)
            x_dict = {k: F.relu(v) for k, v in out.items()}
            return self.head(x_dict["review"]).squeeze(-1)
    return _Model


def _make_prmp_hetero_model(prmp_factory, std_factory, hidden=HIDDEN_DIM):
    """Create a 2-layer hetero GNN with PRMP on child->parent and std on parent->child."""
    class _Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.input_lins = nn.ModuleDict({
                "product": nn.Linear(PRODUCT_FEAT_DIM, hidden),
                "customer": nn.Linear(CUSTOMER_FEAT_DIM, hidden),
                "review": nn.Linear(REVIEW_FEAT_DIM, hidden),
            })
            conv1_dict, conv2_dict = {}, {}
            for k, *_ in EDGE_TYPES:
                fac = prmp_factory if k in PRMP_EDGE_KEYS else std_factory
                conv1_dict[k] = fac(hidden, hidden, hidden)
                conv2_dict[k] = fac(hidden, hidden, hidden)
            self.conv1 = HeteroGNNLayer(conv1_dict)
            self.conv2 = HeteroGNNLayer(conv2_dict)
            self.head = nn.Linear(hidden, 1)
            self.dropout = nn.Dropout(DROPOUT)

        def forward(self, graph):
            x_dict = {
                "product": F.relu(self.input_lins["product"](graph["x_product"])),
                "customer": F.relu(self.input_lins["customer"](graph["x_customer"])),
                "review": F.relu(self.input_lins["review"](graph["x_review"])),
            }
            out = self.conv1(x_dict, graph)
            x_dict = {k: F.relu(self.dropout(v)) for k, v in out.items()}
            out = self.conv2(x_dict, graph)
            x_dict = {k: F.relu(v) for k, v in out.items()}
            return self.head(x_dict["review"]).squeeze(-1)
    return _Model


# Build model classes using factories
_sage_factory = lambda s, d, o: BipartiteSAGEConv(s, d, o)
_gat_factory = lambda s, d, o: BipartiteGATConv(s, d, o, heads=GAT_HEADS)
_prmp_sage_factory = lambda s, d, o: PRMPSAGEConv(s, d, o)
_prmp_gat_factory = lambda s, d, o: PRMPGATConv(s, d, o, heads=GAT_HEADS)

HeteroSAGE = _make_hetero_model(_sage_factory)
HeteroGAT = _make_hetero_model(_gat_factory)
HeteroPRMPSAGE = _make_prmp_hetero_model(_prmp_sage_factory, _sage_factory)
HeteroPRMPGAT = _make_prmp_hetero_model(_prmp_gat_factory, _gat_factory)

VARIANTS = [("SAGE", HeteroSAGE), ("GAT", HeteroGAT),
            ("PRMP_SAGE", HeteroPRMPSAGE), ("PRMP_GAT", HeteroPRMPGAT)]

# Print parameter counts
print(f"{'Variant':<12} {'Total Params':>12} {'Trainable':>12}")
print("-" * 38)
for name, cls in VARIANTS:
    m = cls()
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name:<12} {total:>12,} {trainable:>12,}")
    del m

## Synthetic Graph Construction

We build a small synthetic heterogeneous graph to demonstrate the forward pass and training pipeline. The graph has the same structure as the real Amazon Video Games graph:
- **Review nodes** with 21-dim features (simulating time, text hash, helpful votes)
- **Product nodes** with 5-dim features
- **Customer nodes** with 5-dim features
- Bipartite edges: `review ↔ product` and `review ↔ customer`

In [ ]:
def build_synthetic_graph(n_reviews=N_DEMO_REVIEWS, n_products=N_DEMO_PRODUCTS,
                          n_customers=N_DEMO_CUSTOMERS, seed=DEMO_SEED):
    """Build a synthetic heterogeneous graph for demo purposes."""
    rng = np.random.RandomState(seed)
    torch.manual_seed(seed)

    # Random features
    x_review = torch.randn(n_reviews, REVIEW_FEAT_DIM)
    x_product = torch.randn(n_products, PRODUCT_FEAT_DIM)
    x_customer = torch.randn(n_customers, CUSTOMER_FEAT_DIM)

    # Random ratings (1-5 scale)
    y = torch.tensor(rng.choice([1.0, 2.0, 3.0, 4.0, 5.0], size=n_reviews), dtype=torch.float32)

    # Random edge assignments: each review -> one product, one customer
    review_to_product = torch.tensor(rng.randint(0, n_products, n_reviews), dtype=torch.long)
    review_to_customer = torch.tensor(rng.randint(0, n_customers, n_reviews), dtype=torch.long)
    review_src = torch.arange(n_reviews, dtype=torch.long)

    # Train/val/test split (80/10/10)
    perm = torch.randperm(n_reviews, generator=torch.Generator().manual_seed(seed))
    n_train = int(0.8 * n_reviews)
    n_val = int(0.1 * n_reviews)
    train_mask = perm[:n_train]
    val_mask = perm[n_train:n_train + n_val]
    test_mask = perm[n_train + n_val:]

    graph = {
        "x_product": x_product, "x_customer": x_customer, "x_review": x_review,
        "y": y,
        "edge_review_to_product_src": review_src,
        "edge_review_to_product_dst": review_to_product,
        "edge_product_to_review_src": review_to_product,
        "edge_product_to_review_dst": review_src,
        "edge_review_to_customer_src": review_src,
        "edge_review_to_customer_dst": review_to_customer,
        "edge_customer_to_review_src": review_to_customer,
        "edge_customer_to_review_dst": review_src,
        "train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask,
        "n_products": n_products, "n_customers": n_customers, "n_reviews": n_reviews,
    }
    return graph

graph = build_synthetic_graph()
print(f"Synthetic graph: {graph['n_reviews']} reviews, "
      f"{graph['n_products']} products, {graph['n_customers']} customers")
print(f"Edges: {len(graph['edge_review_to_product_src'])} review->product, "
      f"{len(graph['edge_review_to_customer_src'])} review->customer")
print(f"Split: train={len(graph['train_mask'])}, val={len(graph['val_mask'])}, "
      f"test={len(graph['test_mask'])}")

## Smoke Test: Forward Pass

Verify all 4 model variants produce valid outputs (no NaN/Inf) on the synthetic graph.

In [ ]:
print("Smoke test: single forward pass per variant...")
for name, cls in VARIANTS:
    model = cls().to(DEVICE)
    model.eval()
    with torch.no_grad():
        out = model(graph)
    assert out.shape[0] == graph["n_reviews"], f"{name}: wrong output shape"
    assert not torch.isnan(out).any(), f"{name}: NaN in output"
    assert not torch.isinf(out).any(), f"{name}: Inf in output"
    print(f"  {name}: output shape={out.shape}, mean={out.mean().item():.4f}, std={out.std().item():.4f}")
    del model, out
gc.collect()
print("All variants passed smoke test!")

## Training Demo

Train all 4 variants on the synthetic graph with early stopping. This demonstrates the training pipeline used in the full experiment (which ran 3 seeds x 200 epochs on 50K reviews).

In [ ]:
def train_variant(variant_name, ModelClass, graph, device=DEVICE):
    """Train one model variant and return test metrics."""
    torch.manual_seed(DEMO_SEED)
    np.random.seed(DEMO_SEED)

    model = ModelClass().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()

    best_val_loss = float("inf")
    patience_counter = 0
    best_state = None
    best_epoch = 0
    train_losses, val_losses = [], []

    train_mask = graph["train_mask"]
    val_mask = graph["val_mask"]
    test_mask = graph["test_mask"]
    y = graph["y"]

    t_start = time.time()
    for epoch in range(MAX_EPOCHS):
        model.train()
        optimizer.zero_grad()
        pred = model(graph)
        loss = loss_fn(pred[train_mask], y[train_mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            pred = model(graph)
            val_loss = loss_fn(pred[val_mask], y[val_mask]).item()
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    elapsed = time.time() - t_start

    # Test evaluation
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(graph)
        test_pred = pred[test_mask].cpu().numpy()
        test_true = y[test_mask].cpu().numpy()

    rmse = float(np.sqrt(np.mean((test_pred - test_true) ** 2)))
    mae = float(np.mean(np.abs(test_pred - test_true)))
    ss_res = float(np.sum((test_true - test_pred) ** 2))
    ss_tot = float(np.sum((test_true - test_true.mean()) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else 0.0

    del model, optimizer, best_state
    gc.collect()

    return {
        "rmse": round(rmse, 6), "mae": round(mae, 6), "r2": round(r2, 6),
        "best_epoch": best_epoch, "train_time_s": round(elapsed, 2),
        "train_losses": train_losses, "val_losses": val_losses,
    }

# Train all variants
demo_results = {}
print(f"Training all 4 variants ({MAX_EPOCHS} epochs max, patience={PATIENCE})...\n")
for variant_name, ModelClass in VARIANTS:
    result = train_variant(variant_name, ModelClass, graph)
    demo_results[variant_name] = result
    print(f"  {variant_name:<12} RMSE={result['rmse']:.4f}  MAE={result['mae']:.4f}  "
          f"R2={result['r2']:.4f}  best_epoch={result['best_epoch']}  "
          f"time={result['train_time_s']:.2f}s")

print("\nDemo training complete!")

## Full Experiment Results Visualization

The demo above used a tiny synthetic graph. Below we visualize the **actual experiment results** from the pre-computed data: 50K Amazon Video Games reviews, 3 seeds x 200 epochs, with RMSE/MAE/R2 metrics and pairwise Cohen's d effect sizes.

In [ ]:
# --- Extract pre-computed results from loaded data ---
results_summary = data["metadata"]["results_summary"]
param_counts = data["metadata"]["parameter_counts"]
cohens_d = data["metadata"]["pairwise_cohens_d"]

# --- Print results table ---
print("=" * 75)
print("FULL EXPERIMENT RESULTS (Amazon Video Games, 50K reviews, 3 seeds)")
print("=" * 75)
print(f"\n{'Variant':<12} {'Mean RMSE':>10} {'Std RMSE':>10} {'Mean MAE':>10} "
      f"{'Mean R2':>10} {'Params':>10}")
print("-" * 75)
for variant, summary in results_summary.items():
    print(f"{variant:<12} {summary['mean_rmse']:>10.4f} {summary['std_rmse']:>10.4f} "
          f"{summary['mean_mae']:>10.4f} {summary['mean_r2']:>10.4f} "
          f"{param_counts[variant]['total']:>10,}")

print(f"\n{'Pairwise Cohens d':>30}")
print("-" * 40)
for pair, d_val in cohens_d.items():
    print(f"  {pair:<30} {d_val:>8.4f}")

# --- Visualization ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

variants = list(results_summary.keys())
colors = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']

# 1) RMSE comparison with error bars
ax = axes[0, 0]
means = [results_summary[v]["mean_rmse"] for v in variants]
stds = [results_summary[v]["std_rmse"] for v in variants]
bars = ax.bar(variants, means, yerr=stds, color=colors, alpha=0.8, capsize=5, edgecolor='black', linewidth=0.5)
ax.set_ylabel("Test RMSE")
ax.set_title("Mean Test RMSE by Variant (lower is better)")
ax.set_ylim(min(means) - 0.01, max(means) + 0.01)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{m:.4f}", ha='center', va='bottom', fontsize=9)

# 2) Parameter count comparison
ax = axes[0, 1]
params = [param_counts[v]["total"] for v in variants]
bars = ax.bar(variants, [p/1000 for p in params], color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_ylabel("Parameters (thousands)")
ax.set_title("Model Size by Variant")
for bar, p in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{p//1000}K", ha='center', va='bottom', fontsize=9)

# 3) Per-seed RMSE scatter
ax = axes[1, 0]
for i, v in enumerate(variants):
    seeds_rmse = [r["rmse"] for r in results_summary[v]["per_seed"]]
    seeds = [r["seed"] for r in results_summary[v]["per_seed"]]
    ax.scatter([i]*len(seeds_rmse), seeds_rmse, color=colors[i], s=80, zorder=5, edgecolors='black', linewidth=0.5)
    ax.plot([i]*len(seeds_rmse), seeds_rmse, 'o', color=colors[i], markersize=8)
ax.axhline(y=min(means), color='gray', linestyle='--', alpha=0.5, label=f'Best mean: {min(means):.4f}')
ax.set_xticks(range(len(variants)))
ax.set_xticklabels(variants)
ax.set_ylabel("Test RMSE")
ax.set_title("Per-Seed Test RMSE Distribution")
ax.legend(fontsize=8)

# 4) Cohen's d heatmap
ax = axes[1, 1]
n = len(variants)
d_matrix = np.zeros((n, n))
for i, v1 in enumerate(variants):
    for j, v2 in enumerate(variants):
        key = f"{v1}_vs_{v2}"
        rev_key = f"{v2}_vs_{v1}"
        if key in cohens_d:
            d_matrix[i, j] = cohens_d[key]
        elif rev_key in cohens_d:
            d_matrix[i, j] = -cohens_d[rev_key]

im = ax.imshow(d_matrix, cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(variants, rotation=45, ha='right')
ax.set_yticklabels(variants)
ax.set_title("Pairwise Cohen's d (RMSE)")
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{d_matrix[i,j]:.2f}", ha='center', va='center', fontsize=9,
                color='white' if abs(d_matrix[i,j]) > 1 else 'black')
fig.colorbar(im, ax=ax, shrink=0.8, label="Cohen's d")

plt.tight_layout()
plt.savefig("results_visualization.png", dpi=100, bbox_inches='tight')
plt.show()

print("\nKey findings from the full experiment:")
print("  - GAT provides modest improvement over SAGE (attention vs mean aggregation)")
print("  - PRMP+SAGE comparable to SAGE but with 1.5x parameters")
print("  - PRMP+GAT slightly worse, suggesting residual mechanism has mixed complementarity")
print("  - All variants achieve R2 ~ 0.65, indicating meaningful predictive power")